# Chapter 5 &mdash; Sliding-Window Conditions

**Concept 5 of the Chapter 5 decomposition:** *Sliding-Window Conditions: "Every Block of 3 Has Exactly Two 1s"*

"Every block of 3 has exactly two 1s" &mdash; read <i>every contiguous block</i> correctly, and harvest the vacuous cases.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Sliding-Window/Concept-Sliding-Window.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


"**Every** contiguous block of 3 has exactly two 1s" is a **universally quantified**
condition over overlapping windows, and it has two traps:

* **overlapping** &mdash; positions 1-3, 2-4, 3-5, &hellip;, not a partition into groups of 3;
* **vacuous truth** &mdash; strings of length 0, 1 and 2 have *no* block of 3, so the
  condition holds and they are **in** the language.

Harvest the vacuous cases first: they fix which of your early states are final.

## 2. Definitions

### The specification, written exactly as the English reads

In [ ]:
def in_L(s):
    return all(s[i:i+3].count('1') == 2 for i in range(len(s) - 2))

### The vacuous cases, harvested

In [ ]:
vacuous = ['', '0', '1', '00', '01', '10', '11']
print("in the language vacuously :", [s for s in vacuous if in_L(s)])
print("  -> all 7 -- so the first three 'levels' of states are FINAL")

### The DFA: remember the last two symbols

In [ ]:
# Only IF may start with 'I'.  The other accepting states are named F...
W = md2mc('''DFA
IF   : 0 -> F0      !! nothing seen yet; remember '0'
IF   : 1 -> F1
F0   : 0 -> F00     !! last two are 00
F0   : 1 -> F01
F1   : 0 -> F10
F1   : 1 -> F11
F00  : 0 -> BH      !! 000 has zero 1s -- dead
F00  : 1 -> BH      !! 001 has one 1  -- dead
F01  : 0 -> BH      !! 010 has one 1  -- dead
F01  : 1 -> F11     !! 011 ok; last two now 11
F10  : 0 -> BH      !! 100 dead
F10  : 1 -> F01     !! 101 ok; last two now 01
F11  : 0 -> F10     !! 110 ok; last two now 10
F11  : 1 -> BH      !! 111 has three 1s -- dead
BH   : 0 | 1 -> BH
''')

## 3. Tests

The vacuous strings are accepted &mdash; that is the specification, not a bug.

In [ ]:
for s in vacuous:
    print("%-4r spec=%-5s dfa=%s" % (s, in_L(s), accepts_dfa(W, s)))
assert all(accepts_dfa(W, s) for s in vacuous)

**Overlapping** windows, not a partition: `110110` passes, `110011` does not.

In [ ]:
for s in ['110', '011', '101', '110110', '110011', '111', '000']:
    print("%-8r blocks %-28s spec=%-5s dfa=%s"
          % (s, [s[i:i+3] for i in range(len(s)-2)], in_L(s), accepts_dfa(W, s)))
assert accepts_dfa(W, '110110') and not accepts_dfa(W, '110011')

Agreement with the specification everywhere short.

In [ ]:
from itertools import product
assert all(accepts_dfa(W, ''.join(p)) == in_L(''.join(p))
           for k in range(11) for p in product('01', repeat=k))
print("DFA agrees with the spec on all 2047 strings up to length 10")
print("surviving strings are periodic:",
      [''.join(p) for k in [6] for p in product('01', repeat=k) if in_L(''.join(p))])

## 4. Animation

The three-level fan-out at the start, then a small cycle among 01, 10 and 11.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(W, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Change "exactly two" to "at least two". How does the machine change?
2. Redo it for blocks of 4. How many states before minimization?
3. Why does misreading "every block" as a *partition* give a different language?

In [ ]:
# Your work for the exercises above.